# 01 一维热传导方程的有限差分解

本 notebook 用有限差分法（Finite Difference Method, FDM）数值求解一维热传导方程，并做可视化与稳定性分析。

## 方程

考虑一维热方程初边值问题：

$$
\begin{cases}
u_t = \alpha u_{xx}, & x \in (0, L), \, t > 0 \\
u(0, t) = 0, \quad u(L, t) = 0, & t \ge 0 \\
u(x, 0) = u_0(x), & x \in [0, L]
\end{cases}
$$

其中 $\alpha > 0$ 为热扩散系数。

## 方法：显式欧拉（Forward Euler）+ 中心差分

空间用二阶中心差分离散 $u_{xx}$，时间用一阶向前差分离散 $u_t$。

记 $u_i^n \approx u(x_i, t_n)$，其中 $x_i = i\Delta x$，$t_n = n\Delta t$。

离散格式：

$$
\frac{u_i^{n+1} - u_i^n}{\Delta t} = \alpha \frac{u_{i+1}^n - 2u_i^n + u_{i-1}^n}{\Delta x^2}
$$

整理得递推公式：

$$
u_i^{n+1} = u_i^n + r(u_{i+1}^n - 2u_i^n + u_{i-1}^n)
$$

其中 $r = \alpha \cdot \frac{\Delta t}{\Delta x^2}$。

## 稳定性条件（CFL / von Neumann 分析）

显式欧拉的稳定性条件为：

$$r = \alpha \frac{\Delta t}{\Delta x^2} \le \frac{1}{2}$$

即 $\Delta t \le \frac{\Delta x^2}{2\alpha}$。不满足时数值解会振荡发散。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

%matplotlib inline

## 1. 参数设置

In [ ]:
alpha = 0.01   # 热扩散系数
L = 1.0        # 空间域长度
T = 1.0        # 总时间
nx = 50        # 空间格点数
nt = 1000      # 时间步数

dx = L / (nx - 1)
dt = T / nt
r = alpha * dt / dx**2

print(f"dx = {dx:.4f}")
print(f"dt = {dt:.6f}")
print(f"r  = {r:.4f}")
print(f"稳定性条件 r <= 0.5 ? {'满足' if r <= 0.5 else '不满足'}")

## 2. 初始条件

取初始温度分布为一个正弦波（正好是解析解的形式，方便对比）：

$$u_0(x) = \sin(\pi x)$$

对应的解析解为：

$$u(x, t) = e^{-\alpha \pi^2 t} \sin(\pi x)$$

In [ ]:
x = np.linspace(0, L, nx)
u0 = np.sin(np.pi * x)  # 初始条件

plt.figure(figsize=(8, 4))
plt.plot(x, u0, 'b-', linewidth=2)
plt.xlabel('x')
plt.ylabel('u(x, 0)')
plt.title('初始温度分布')
plt.grid(True, alpha=0.3)
plt.show()

## 3. 显式欧拉求解

In [ ]:
def solve_heat_explicit(u0, alpha, dx, dt, nt):
    """显式欧拉法求解一维热方程。"""
    nx = len(u0)
    r = alpha * dt / dx**2
    u = u0.copy()
    history = [u.copy()]
    
    for n in range(nt):
        u_new = u.copy()
        # 内部点更新（边界保持为 0）
        u_new[1:-1] = u[1:-1] + r * (u[2:] - 2*u[1:-1] + u[:-2])
        u = u_new
        history.append(u.copy())
    
    return np.array(history)

In [ ]:
u_history = solve_heat_explicit(u0, alpha, dx, dt, nt)
print(f"计算完成，共 {nt+1} 个时间步")
print(f"最终时刻 max(u) = {np.max(u_history[-1]):.6f}")

## 4. 与解析解对比

In [ ]:
def analytical_solution(x, t, alpha):
    return np.exp(-alpha * np.pi**2 * t) * np.sin(np.pi * x)

# 几个时间点对比
t_checkpoints = [0, 0.2, 0.5, 1.0]

plt.figure(figsize=(10, 6))
for t_val in t_checkpoints:
    n_step = int(t_val / dt)
    n_step = min(n_step, nt)
    t_actual = n_step * dt
    u_num = u_history[n_step]
    u_exact = analytical_solution(x, t_actual, alpha)
    l2_err = np.sqrt(np.mean((u_num - u_exact)**2))
    
    line, = plt.plot(x, u_num, '-', linewidth=2, label=f't={t_val:.1f} (数值)')
    plt.plot(x, u_exact, '--', color=line.get_color(), linewidth=1.5, label=f't={t_val:.1f} (解析)')
    print(f"t={t_val:.1f}: L2 误差 = {l2_err:.2e}")

plt.xlabel('x')
plt.ylabel('u(x, t)')
plt.title('数值解 vs 解析解')
plt.legend(loc='upper right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.show()

## 5. 收敛性验证

理论上，显式欧拉对时间是一阶收敛，对空间是二阶收敛。我们来验证一下：
固定 $r = 0.25$（满足稳定性），让 $\Delta x \to 0$，观察 $\Delta t$ 也随之减小，误差应该以 $O(\Delta x^2)$ 的速率收敛。

In [ ]:
r_fixed = 0.25  # 固定 r，保证稳定性
nx_list = [10, 20, 40, 80, 160]
errors_l2 = []
errors_linf = []
dx_list = []

for nx_val in nx_list:
    dx_val = L / (nx_val - 1)
    dt_val = r_fixed * dx_val**2 / alpha
    nt_val = int(T / dt_val)
    
    x_val = np.linspace(0, L, nx_val)
    u0_val = np.sin(np.pi * x_val)
    
    u_hist = solve_heat_explicit(u0_val, alpha, dx_val, dt_val, nt_val)
    u_num = u_hist[-1]
    u_exact = analytical_solution(x_val, nt_val * dt_val, alpha)
    
    l2_err = np.sqrt(np.mean((u_num - u_exact)**2))
    linf_err = np.max(np.abs(u_num - u_exact))
    
    errors_l2.append(l2_err)
    errors_linf.append(linf_err)
    dx_list.append(dx_val)
    print(f"nx={nx_val:3d}, dx={dx_val:.4f}, L2 err={l2_err:.2e}, L∞ err={linf_err:.2e}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.loglog(dx_list, errors_l2, 'bo-', label='L2 误差', linewidth=2, markersize=8)
plt.loglog(dx_list, errors_linf, 'rs-', label='L∞ 误差', linewidth=2, markersize=8)

# 画二阶收敛参考线
ref_x = np.array(dx_list)
ref_y = 0.1 * ref_x**2
plt.loglog(ref_x, ref_y, 'k--', label='O(Δx²) 参考', alpha=0.7)

plt.xlabel('Δx')
plt.ylabel('误差')
plt.title('收敛性验证（固定 r=0.25）')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.gca().invert_xaxis()
plt.show()

## 6. 稳定性演示：当 r > 0.5 时会怎样？

故意取一个不稳定的参数，看看数值解怎么爆炸的。

In [ ]:
nx_unstable = 50
dx_unstable = L / (nx_unstable - 1)
r_unstable = 0.6  # > 0.5，不稳定
dt_unstable = r_unstable * dx_unstable**2 / alpha
nt_unstable = 50  # 几步就炸了

x_unstable = np.linspace(0, L, nx_unstable)
u0_unstable = np.sin(np.pi * x_unstable)

u_unstable_hist = solve_heat_explicit(u0_unstable, alpha, dx_unstable, dt_unstable, nt_unstable)

print(f"r = {r_unstable}（> 0.5，不稳定）")
print(f"初始 max = {np.max(np.abs(u_unstable_hist[0])):.4f}")
print(f"第 {nt_unstable} 步 max = {np.max(np.abs(u_unstable_hist[-1])):.4f}")

plt.figure(figsize=(10, 6))
for n in [0, 5, 10, 20, 50]:
    if n <= nt_unstable:
        plt.plot(x_unstable, u_unstable_hist[n], label=f't={n*dt_unstable:.4f}')
plt.xlabel('x')
plt.ylabel('u')
plt.title(f'不稳定情形：r = {r_unstable} > 0.5')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.show()

## 7. 隐式欧拉（Backward Euler）——无条件稳定

隐式格式：

$$
\frac{u_i^{n+1} - u_i^n}{\Delta t} = \alpha \frac{u_{i+1}^{n+1} - 2u_i^{n+1} + u_{i-1}^{n+1}}{\Delta x^2}
$$

整理成线性方程组 $A u^{n+1} = u^n$，其中 $A$ 是三对角矩阵。

隐式欧拉是**无条件稳定**的，$r$ 多大都不会炸（但精度可能差）。

In [ ]:
def solve_heat_implicit(u0, alpha, dx, dt, nt):
    """隐式欧拉法求解一维热方程（用 Thomas 算法解三对角方程组）。"""
    nx = len(u0)
    r = alpha * dt / dx**2
    u = u0.copy()
    history = [u.copy()]
    
    # 三对角矩阵：-r*u_{i-1} + (1+2r)*u_i - r*u_{i+1} = u_i^n
    a = -r * np.ones(nx - 2)   # 下对角线
    b = (1 + 2*r) * np.ones(nx - 2)  # 主对角线
    c = -r * np.ones(nx - 2)   # 上对角线
    
    for _ in range(nt):
        d = u[1:-1].copy()  # 右端项
        
        # Thomas 算法
        n_tri = len(d)
        cp = np.zeros(n_tri)
        dp = np.zeros(n_tri)
        
        cp[0] = c[0] / b[0]
        dp[0] = d[0] / b[0]
        
        for i in range(1, n_tri):
            m = b[i] - a[i] * cp[i-1]
            cp[i] = c[i] / m
            dp[i] = (d[i] - a[i] * dp[i-1]) / m
        
        u_new = np.zeros(nx)
        u_new[-2] = dp[-1]
        for i in range(n_tri-2, -1, -1):
            u_new[i+1] = dp[i] - cp[i] * u_new[i+2]
        
        # 边界条件保持为 0
        u_new[0] = 0
        u_new[-1] = 0
        u = u_new
        history.append(u.copy())
    
    return np.array(history)

In [ ]:
# 用刚才那个不稳定的参数试试隐式格式
u_impl_hist = solve_heat_implicit(u0_unstable, alpha, dx_unstable, dt_unstable, nt_unstable)

print(f"r = {r_unstable}（隐式格式，无条件稳定）")
print(f"初始 max = {np.max(np.abs(u_impl_hist[0])):.4f}")
print(f"第 {nt_unstable} 步 max = {np.max(np.abs(u_impl_hist[-1])):.6f}")

t_final = nt_unstable * dt_unstable
u_exact_final = analytical_solution(x_unstable, t_final, alpha)
l2_err_impl = np.sqrt(np.mean((u_impl_hist[-1] - u_exact_final)**2))
print(f"L2 误差（隐式）= {l2_err_impl:.2e}")

plt.figure(figsize=(10, 6))
plt.plot(x_unstable, u_impl_hist[0], 'k--', label='t=0')
plt.plot(x_unstable, u_impl_hist[-1], 'b-', linewidth=2, label=f't={t_final:.4f} (隐式)')
plt.plot(x_unstable, u_exact_final, 'r--', linewidth=1.5, label='解析解')
plt.xlabel('x')
plt.ylabel('u')
plt.title(f'隐式欧拉：r = {r_unstable}，依然稳定')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 小结

| 方法 | 稳定性 | 精度（时间） | 精度（空间） | 每步计算量 |
|---|---|---|---|---|
| 显式欧拉 | 条件稳定（$r \le 1/2$） | $O(\Delta t)$ | $O(\Delta x^2)$ | $O(n)$ |
| 隐式欧拉 | 无条件稳定 | $O(\Delta t)$ | $O(\Delta x^2)$ | $O(n)$（三对角） |
| Crank-Nicolson | 无条件稳定 | $O(\Delta t^2)$ | $O(\Delta x^2)$ | $O(n)$（三对角） |

显式格式简单但受限于稳定性条件（时间步要很小）；隐式格式稳定但每步要解方程组；Crank-Nicolson 是两者折中，时间二阶精度且无条件稳定。